<a href="https://colab.research.google.com/github/G-Yadidhya/notebook/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/G-Yadidhya/notebook/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring  
**Decision moment:** An editor is deciding which content items to review at the end of the current snapshot.  
**Data:** The public-safe, anonymized starter dataset (`content_refresh_anonymized.csv`, 30,000 rows). This notebook does not need the gated warehouse or a Hugging Face token.

This is a transparent **decision-support baseline**, not a prediction that a refresh will cause traffic to increase. It uses only snapshot signals available for review: last-update age, 90-day impressions, average position, and CTR. It does not use `trend_direction`, `trend_pct`, or any future-window label as a scoring input.


## 1. Check two signals before writing the rule

### Signal A — staleness and visibility

A refresh flag should not treat age alone as enough. The bucket table reports the number of items (`n`), median 90-day impressions, and how many items clear a 500-impression visibility floor.

**Verdict: OPPOSITE.** The oldest bucket is small and has very low median impressions, so “old” by itself is not a useful priority rule. The baseline must require observable demand as well as staleness.

### Signal B — CTR relative to search position (flag-linked)

FlyRank's CTR-fix logic considers CTR alongside visibility and position. The table below uses items with at least 500 impressions and a known average position; it reports `n`, median CTR, and the share below 0.5% CTR. In this dataset, CTR generally falls and low-CTR prevalence rises as position gets worse.

**Verdict: CONFIRMED, with a caveat.** CTR is position-dependent, so low CTR should be reviewed in context. A low CTR is not proof that the title/snippet is the cause; query mix and SERP features may explain it.


In [7]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Work from a checked-out repository when available; otherwise download the public CSV.
possible_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next((p for p in possible_roots if (p / "data/raw/content_refresh_anonymized.csv").exists()), None)

if repo_root is None:
    repo_root = Path.cwd()
    csv_path = repo_root / "data/raw/content_refresh_anonymized.csv"
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    urls = [
        "https://raw.githubusercontent.com/G-Yadidhya/notebook/main/data/raw/content_refresh_anonymized.csv",
        "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv",
    ]
    last_error = None
    for url in urls:
        try:
            downloaded = pd.read_csv(url)
            downloaded.to_csv(csv_path, index=False)
            break
        except Exception as exc:
            last_error = exc
    else:
        raise RuntimeError(
            "Could not find the starter CSV locally or download it. Open the full repository in Colab, "
            "or make sure the repository is public and internet access is enabled."
        ) from last_error
else:
    csv_path = repo_root / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)
required = {
    "content_id", "client_id", "impressions_90d", "ctr", "avg_position",
    "days_since_last_update", "content_age_days"
}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"Starter dataset is missing required columns: {missing}")

print(f"Dataset: {csv_path}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print("This audit uses only snapshot columns; trend_direction and trend_pct are not scoring inputs.")


Dataset: /content/data/raw/content_refresh_anonymized.csv
Rows: 30,000 | Columns: 44
This audit uses only snapshot columns; trend_direction and trend_pct are not scoring inputs.


### Signal A bucket table — staleness

Age buckets are descriptive checks, not labels. The `n` column is printed explicitly so a small bucket is not mistaken for strong evidence.


In [8]:
age_bins = [-1, 30, 90, 180, np.inf]
age_labels = ["0–30 days", "31–90 days", "91–180 days", "181+ days"]
age_audit = df.copy()
age_audit["update_age_bucket"] = pd.cut(
    age_audit["days_since_last_update"], bins=age_bins, labels=age_labels
)
age_table = (
    age_audit.groupby("update_age_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_impressions_90d=("impressions_90d", "median"),
        visible_items_n=("impressions_90d", lambda s: int((s >= 500).sum())),
    )
    .reset_index()
)
age_table["visible_share_pct"] = (100 * age_table["visible_items_n"] / age_table["n"]).round(1)
display(age_table)


,update_age_bucket,n,median_impressions_90d,visible_items_n,visible_share_pct
0,0–30 days,20480,470.0,10063,49.1
1,31–90 days,175,510.0,88,50.3
2,91–180 days,9171,1692.0,6558,71.5
3,181+ days,174,15.5,17,9.8


### Signal B bucket table — CTR by position (flag-linked)

`avg_position = 0` means no position data, so those rows are excluded from this table. CTR values are percentage points (`0.5` means 0.5%), not proportions like `0.5 = 50%`.


In [9]:
position_audit = df.loc[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)].copy()
position_audit["position_bucket"] = pd.cut(
    position_audit["avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["1–3", "4–10", "11–20", "21–50", "50+"],
    include_lowest=True,
)
position_table = (
    position_audit.groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_ctr_pct=("ctr", "median"),
        low_ctr_share_pct=("ctr", lambda s: 100 * (s < 0.5).mean()),
        median_impressions_90d=("impressions_90d", "median"),
    )
    .reset_index()
)
position_table["median_ctr_pct"] = position_table["median_ctr_pct"].round(3)
position_table["low_ctr_share_pct"] = position_table["low_ctr_share_pct"].round(1)
display(position_table)


,position_bucket,n,median_ctr_pct,low_ctr_share_pct,median_impressions_90d
0,1–3,480,0.20,75.0,4382.5
1,4–10,7084,0.24,79.2,4502.0
2,11–20,4459,0.17,85.0,2166.0
3,21–50,4314,0.09,95.0,2459.0
4,50+,389,0.00,99.2,965.0


## 2. Encode ONE rule and write the ranked queue

**Rule in plain words:** prioritize a page for combined refresh/snippet review only when it is stale (at least 180 days since update), has at least 500 impressions in the trailing 90 days, has a known average position in the top 20, and CTR is below 0.5%. This combines age with demand and a position-aware CTR opportunity; age alone is not enough.

**One reason code:** `stale_visible_low_ctr_page` for every selected page. Rows that do not qualify have a blank reason code and action `not_selected`.

**Score:** for selected pages, `log(1 + impressions_90d) × (0.5 − ctr)`. The score ranks qualifying pages by visible demand and the gap below the CTR threshold. It is a prioritization heuristic, not a calibrated probability.

The queue is written to `work/outputs/baseline_action_score.csv` on every run. The CSV is a generated artifact and should stay out of Git; the notebook regenerates it.


In [10]:
# The rule uses only point-in-time snapshot columns; no trend/label columns are referenced.
score_inputs = ["days_since_last_update", "impressions_90d", "avg_position", "ctr"]
rule_mask = (
    (df["days_since_last_update"] >= 180)
    & (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)

queue = df[["content_id", "client_id", "content_type", *score_inputs]].copy()
queue["score"] = 0.0
queue.loc[rule_mask, "score"] = (
    np.log1p(queue.loc[rule_mask, "impressions_90d"])
    * (0.5 - queue.loc[rule_mask, "ctr"])
)
queue["reason_code"] = ""
queue.loc[rule_mask, "reason_code"] = "stale_visible_low_ctr_page"
queue["action"] = np.where(rule_mask, "review_refresh_and_snippet", "not_selected")
queue = queue.sort_values(["score", "impressions_90d"], ascending=False, kind="mergesort").reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

output_dir = repo_root / "work/outputs"
output_dir.mkdir(parents=True, exist_ok=True)
queue_path = output_dir / "baseline_action_score.csv"
queue.to_csv(queue_path, index=False)

selected_n = int(rule_mask.sum())
print(f"Selected for review: {selected_n:,} of {len(df):,} rows")
print(f"Wrote ranked queue: {queue_path}")
print("Positive reason codes:", sorted(queue.loc[queue["reason_code"].ne(""), "reason_code"].unique().tolist()))
display(queue.head(10))


Selected for review: 10 of 30,000 rows
Wrote ranked queue: /content/work/outputs/baseline_action_score.csv
Positive reason codes: ['stale_visible_low_ctr_page']


,rank,content_id,client_id,content_type,days_since_last_update,impressions_90d,avg_position,ctr,score,reason_code,action
0,1,content_cf56e2e2e282,client_7f2253d7e2,keyword article,194,61678,19.7,0.15,3.860395,stale_visible_low_ctr_page,review_refresh_and_snippet
1,2,content_928af3e22c80,client_7f2253d7e2,keyword article,193,1697,15.8,0.12,2.826138,stale_visible_low_ctr_page,review_refresh_and_snippet
2,3,content_c2d929d83eaa,client_7f2253d7e2,keyword article,193,7558,17.9,0.20,2.679148,stale_visible_low_ctr_page,review_refresh_and_snippet
3,4,content_6226ee6adc91,client_d029fa3a95,keyword article,183,545,17.8,0.18,2.016838,stale_visible_low_ctr_page,review_refresh_and_snippet
4,5,content_77d4d5930e5e,client_7f2253d7e2,keyword article,194,828,18.6,0.24,1.747257,stale_visible_low_ctr_page,review_refresh_and_snippet
5,6,content_72496874f806,client_4ec9599fc2,keyword article,301,821,5.8,0.24,1.745053,stale_visible_low_ctr_page,review_refresh_and_snippet
6,7,content_e3ff1b093148,client_d029fa3a95,keyword article,183,1408,7.8,0.28,1.595140,stale_visible_low_ctr_page,review_refresh_and_snippet
7,8,content_fe16a55cd13d,client_7f2253d7e2,keyword article,194,4556,16.4,0.33,1.432151,stale_visible_low_ctr_page,review_refresh_and_snippet
8,9,content_7f116ae1f6f5,client_9400f1b21c,keyword article,301,954,9.0,0.42,0.548937,stale_visible_low_ctr_page,review_refresh_and_snippet
9,10,content_0a91db491d14,client_7f2253d7e2,keyword article,193,13299,10.5,0.49,0.094955,stale_visible_low_ctr_page,review_refresh_and_snippet


## 3. Top-10 review

Each row below has an action, the exact reason it entered the queue, and a concrete way the recommendation could be wrong. The identifiers are pseudonymous. These are candidates for human review, not automatic changes.


In [11]:
top10 = queue.loc[queue["action"].eq("review_refresh_and_snippet")].head(10).copy()

def why_selected(row):
    return (
        f"{int(row['days_since_last_update'])} days since update; "
        f"{int(row['impressions_90d']):,} impressions/90d; "
        f"position {row['avg_position']:.1f}; CTR {row['ctr']:.2f}% (<0.5%)."
    )

def what_could_make_wrong(row):
    notes = []
    if row["avg_position"] > 15:
        notes.append("position is near the edge of the top-20 rule and query mix may explain low CTR")
    else:
        notes.append("SERP features or query intent may explain CTR better than the snippet")
    if row["ctr"] >= 0.4:
        notes.append("CTR is close to the 0.5% cutoff, so small measurement changes could remove eligibility")
    if row["impressions_90d"] < 1500:
        notes.append("volume is modest and may vary across periods")
    if row["days_since_last_update"] < 190:
        notes.append("age is only just beyond the 180-day threshold")
    notes.append("a human should inspect the query mix and page before acting")
    return "; ".join(notes) + "."

top10["why_it_is_here"] = top10.apply(why_selected, axis=1)
top10["confidence_note"] = "Rule fit is clear; business value is unverified."
top10["what_would_make_it_wrong"] = top10.apply(what_could_make_wrong, axis=1)
review = top10[["rank", "content_id", "action", "reason_code", "score", "why_it_is_here", "confidence_note", "what_would_make_it_wrong"]]
if len(review) < 10:
    print(f"Only {len(review)} items met the rule; review all selected items and do not invent more candidates.")
display(review)


,rank,content_id,action,reason_code,score,why_it_is_here,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,review_refresh_and_snippet,stale_visible_low_ctr_page,3.860395,"194 days since update; 61,678 impressions/90d;...",Rule fit is clear; business value is unverified.,position is near the edge of the top-20 rule a...
1,2,content_928af3e22c80,review_refresh_and_snippet,stale_visible_low_ctr_page,2.826138,"193 days since update; 1,697 impressions/90d; ...",Rule fit is clear; business value is unverified.,position is near the edge of the top-20 rule a...
2,3,content_c2d929d83eaa,review_refresh_and_snippet,stale_visible_low_ctr_page,2.679148,"193 days since update; 7,558 impressions/90d; ...",Rule fit is clear; business value is unverified.,position is near the edge of the top-20 rule a...
3,4,content_6226ee6adc91,review_refresh_and_snippet,stale_visible_low_ctr_page,2.016838,183 days since update; 545 impressions/90d; po...,Rule fit is clear; business value is unverified.,position is near the edge of the top-20 rule a...
4,5,content_77d4d5930e5e,review_refresh_and_snippet,stale_visible_low_ctr_page,1.747257,194 days since update; 828 impressions/90d; po...,Rule fit is clear; business value is unverified.,position is near the edge of the top-20 rule a...
5,6,content_72496874f806,review_refresh_and_snippet,stale_visible_low_ctr_page,1.745053,301 days since update; 821 impressions/90d; po...,Rule fit is clear; business value is unverified.,SERP features or query intent may explain CTR ...
6,7,content_e3ff1b093148,review_refresh_and_snippet,stale_visible_low_ctr_page,1.595140,"183 days since update; 1,408 impressions/90d; ...",Rule fit is clear; business value is unverified.,SERP features or query intent may explain CTR ...
7,8,content_fe16a55cd13d,review_refresh_and_snippet,stale_visible_low_ctr_page,1.432151,"194 days since update; 4,556 impressions/90d; ...",Rule fit is clear; business value is unverified.,position is near the edge of the top-20 rule a...
8,9,content_7f116ae1f6f5,review_refresh_and_snippet,stale_visible_low_ctr_page,0.548937,301 days since update; 954 impressions/90d; po...,Rule fit is clear; business value is unverified.,SERP features or query intent may explain CTR ...
9,10,content_0a91db491d14,review_refresh_and_snippet,stale_visible_low_ctr_page,0.094955,"193 days since update; 13,299 impressions/90d;...",Rule fit is clear; business value is unverified.,SERP features or query intent may explain CTR ...


## 4. Weak picks and leakage check

**Weak-pick lesson:** the lowest-scoring selected pages are still selected because they pass the thresholds, but their scores may be only slightly above the CTR cutoff or their impressions may be modest. A hard threshold can create brittle picks; inspect the query-level context before spending editorial time.

**Leakage check:** the scoring formula uses only `days_since_last_update`, `impressions_90d`, `avg_position`, and `ctr`. It does not use `trend_direction`, `trend_pct`, `is_declining_label`, a future time window, or a product-generated flag. The two signal audits are descriptive bucket summaries; they do not feed into the score as labels.

**Named limitation:** the starter dataset is a single anonymized snapshot with aggregate 90-day metrics. It cannot show whether a content edit causes recovery, and aggregate position/CTR can hide differences across queries, devices, and SERP layouts.


In [12]:
# Show the weakest selected picks so the reviewer can challenge the rule.
weak_picks = queue.loc[queue["action"].eq("review_refresh_and_snippet")].tail(3).sort_values("score")
display(weak_picks[["rank", "content_id", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "score", "reason_code", "action"]])

forbidden_inputs = {"trend_direction", "trend_pct", "is_declining_label", "future_ctr", "future_impressions"}
assert forbidden_inputs.isdisjoint(score_inputs), "Leakage: a forbidden label/future field entered the score inputs."
assert queue.loc[queue["reason_code"].ne(""), "reason_code"].nunique() <= 1
assert queue.loc[queue["action"].eq("review_refresh_and_snippet"), "score"].gt(0).all()

metrics = {
    "lane": "Refresh / Content Opportunity Scoring",
    "dataset_rows": int(len(df)),
    "selected_rows": selected_n,
    "selection_rate": round(selected_n / len(df), 6),
    "rule": {
        "days_since_last_update_min": 180,
        "impressions_90d_min": 500,
        "avg_position_min_exclusive": 0,
        "avg_position_max_inclusive": 20,
        "ctr_max_exclusive_pct": 0.5,
        "score": "log1p(impressions_90d) * (0.5 - ctr)",
        "reason_code": "stale_visible_low_ctr_page",
    },
    "score_inputs": score_inputs,
    "forbidden_label_or_future_inputs_used": False,
    "limitations": ["single anonymized snapshot", "aggregate CTR and position hide query-level context", "not causal evidence"],
}
metrics_path = output_dir / "baseline_score_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"Leakage assertions passed. Metrics receipt written to: {metrics_path}")
print("Remember: commit the notebook and JSON receipt; do not commit baseline_action_score.csv.")


,rank,content_id,days_since_last_update,impressions_90d,avg_position,ctr,score,reason_code,action
9,10,content_0a91db491d14,193,13299,10.5,0.49,0.094955,stale_visible_low_ctr_page,review_refresh_and_snippet
8,9,content_7f116ae1f6f5,301,954,9.0,0.42,0.548937,stale_visible_low_ctr_page,review_refresh_and_snippet
7,8,content_fe16a55cd13d,194,4556,16.4,0.33,1.432151,stale_visible_low_ctr_page,review_refresh_and_snippet


Leakage assertions passed. Metrics receipt written to: /content/work/outputs/baseline_score_metrics.json
Remember: commit the notebook and JSON receipt; do not commit baseline_action_score.csv.


## 5. Self-check

- [x] Two signal checks have visible bucket tables with `n` and one-word verdicts.
- [x] At least one signal is linked to FlyRank's CTR-fix logic.
- [x] One transparent rule, one positive reason code, a score, and an action label.
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv`.
- [x] Top-10 review includes action, why selected, confidence, and what would make it wrong.
- [x] No future-window or label-derived inputs in the score.
- [x] Named limitation included.

